1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
    - 토큰수 초과로 답변을 생성하지 못할 수 있고
    - 문서가 길면 (인풋이 길면) 답변 생성이 오래 걸림
3. 임베딩(텍스트를 숫자 벡터로 변환) -> 벡터 데이터베이스에 저장
4. 질문이 있을 대, 벡터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달달

1. 문서의 내용을 읽는다.

In [ ]:
%pip install -qU  docx2txt langchain-community

In [9]:
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader('./tax.docx')


2. 문서의 내용을 쪼갠다.

In [7]:
%pip install -qU langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, TextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 하나의 청크가 가질 수 있는 토큰 수수
    chunk_overlap=200, # 청크간 겹치는 정도, 유사도 검사를 할 때 우너하는 정보를 가져올 확률을 높인다.
)

document_list = loader.load_and_split(text_splitter=text_splitter)

3. 쪼개진 문서를 임베딩한다. 이후 벡터 데이터베이스에 저장한다.

In [ ]:
%pip install langchain-huggingface sentence-transformers

In [ ]:
!ollama pull nomic-embed-text
%pip install langchain-ollama

In [ ]:
%pip install python-dotenv langchain-upstage

In [3]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

load_dotenv()

upstage_embedding = UpstageEmbeddings(model="solar-embedding-1-large")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

hugging_embedding = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")

In [ ]:
from langchain_ollama import OllamaEmbeddings

ollama_embedding = OllamaEmbeddings(model="nomic-embed-text")

In [ ]:
%pip install -qU "langchain-chroma>=0.1.2"

In [11]:
from langchain_chroma import Chroma

database = Chroma.from_documents(documents= document_list, embedding= upstage_embedding, collection_name= 'chroma-tax', persist_directory="./chroma")

In [12]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
retrieved_docs = database.similarity_search(query, k=10)

In [ ]:
retrieved_docs

In [13]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-haiku-4-5")

In [14]:
prompt = f"""[Identity]
- 당신은 최고의 한국 소득세 전문가 입니다.
- [Context]를 참고해서 사용자의 질문에 답변해주세요

[Context]
{retrieved_docs}

Question: {query}
"""

In [15]:
ai_message = llm.invoke(prompt)

In [16]:
ai_message.content

'# 연봉 5천만원 직장인의 소득세 계산\n\n제공하신 세법 문서를 바탕으로 기본적인 계산 과정을 설명하겠습니다.\n\n## 1단계: 근로소득공제 계산\n\n**총급여액: 5,000만원**\n\n제47조에 따른 근로소득공제:\n- 총급여액이 5,000만원이므로 공제액 계산표를 적용하면 약 **1,540만원**이 공제됩니다.\n- (정확한 계산표는 문서에 미제시되었으나, 일반적으로 이 수준입니다)\n\n**근로소득금액 = 5,000만원 - 1,540만원 = 3,460만원**\n\n## 2단계: 종합소득과세표준 계산\n\n기본공제(제50조):\n- 본인: 150만원\n- 배우자/부양가족이 있다면 추가 공제 적용\n\n**과세표준 = 근로소득금액 - 인적공제**\n\n## 3단계: 세액 계산\n\n제55조의 세율 적용:\n- 과세표준에 따라 누진세율 적용\n- 5천만원대 소득은 **24%~35%** 구간 적용\n\n## 4단계: 세액공제\n\n- 근로소득세액공제(제59조): 총급여액 기준으로 공제\n- 5,000만원 급여의 경우 **66만원** 공제\n\n---\n\n## ⚠️ 정확한 계산을 위해 필요한 정보\n\n실제 세액 계산을 위해서는:\n- ✓ 가족 상황 (배우자, 부양가족 수)\n- ✓ 기타 소득 여부\n- ✓ 세액공제 대상 항목 (보험료, 의료비, 자녀세액공제 등)\n- ✓ 2024년 정확한 공제액 표\n\n**정확한 계산은 세무사나 국세청 홈택스(hometax.go.kr)를 통해 확인하시기를 권장합니다.**'